# TASK 05 — PhoBERT Vietnamese Text Processing & Embedding Extraction

**Project:** SecondLife AI Pricing  
**Domain:** Pre-owned Household Appliances in Vietnam  
**Target Architecture:** Multimodal Feature Extraction (PhoBERT 768-d Text + CLIP 512-d Vision + Tabular Features)  
**Model:** `vinai/phobert-base` (Pretrained Vietnamese RoBERTa Backbone)  
**Expected Dimension:** 768 float32 contextual representation  
**Objective:** Build a robust, reproducible Vietnamese text preprocessing and feature extraction pipeline for appliance titles and descriptions, preserving technical entities, warranty conditions, and model numbers.  

> **STRICT GOVERNANCE:**  
> - **Pretrained Feature Extraction Only:** Zero model fine-tuning. Weights remain frozen in `torch.inference_mode()`.  
> - **No Fake Prices:** Fixtures contain appliance technical specs and condition metadata only. No synthetic VND listing prices.  
> - **Training Blocked:** Price regression training remains BLOCKED pending authorized Chợ Tốt / SecondLife real Vietnamese listing data.

## 1. Environment Setup & Hardware Diagnostics

We detect the available compute environment (CUDA GPU on Kaggle/Colab or CPU locally) and verify library versions.

In [ ]:
import os
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from transformers import AutoTokenizer, AutoModel

# Ensure project root is in path
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.features.vietnamese_text_processor import VietnameseTextProcessor, SegmentationEngine
from src.features.extract_phobert_embeddings import PhoBertFeatureExtractor, run_extraction_pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch Version: {torch.__version__}")
print(f"Execution Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Running on multi-core CPU inference mode.")

## 2. Vietnamese Text Processor Demonstration

Vietnamese text preprocessing for PhoBERT requires special handling:
1. **Unicode NFC Normalization:** Resolves decomposed/composed diacritics (e.g. `u\u0309` vs `ủ`).
2. **Marketplace Abbreviation Expansion:** Expands common classified terms (`bh` -> `bảo hành`, `inv` -> `inverter`, `đmx` -> `Điện Máy Xanh`).
3. **Technical Entity Protection:** Shields hyphenated model codes (`NR-BV280QSVN`, `EWF8025DGWA`) and measurement units (`9kg`, `255L`, `220V`) from being fragmented by Vietnamese word segmenters.
4. **Word Segmentation:** Produces underscore-linked compound words (e.g. `tủ_lạnh`, `máy_giặt`, `tiết_kiệm_điện`) required by PhoBERT's vocabulary.

In [ ]:
processor = VietnameseTextProcessor(segmenter=SegmentationEngine.PYVI, expand_abbreviations=True)

raw_example_title = "TL panasonic Inverter 255l NR-BV280QSVN bh 6th"
raw_example_desc = "Máy chạy cực êm, tiết kiệm điện, còn bh đmx 6 tháng. Giá thanh lý."

processed = processor.process_listing(raw_example_title, raw_example_desc)

print("=== PREPROCESSING PIPELINE BREAKDOWN ===")
print(f"Raw Title:            {raw_example_title}")
print(f"Raw Desc:             {raw_example_desc}")
print(f"Normalized Title:     {processed['title_normalized']}")
print(f"Normalized Desc:      {processed['description_normalized']}")
print(f"Segmented Title:      {processed['title_segmented']}")
print(f"Segmented Desc:       {processed['description_segmented']}")
print(f"Combined for PhoBERT: {processed['combined_segmented']}")
print("\nTechnical Model Preservation Test:")
assert "NR-BV280QSVN" in processed['combined_segmented'], "Error: Model number was fragmented!"
print("PASSED: Technical model 'NR-BV280QSVN' preserved intact without fragmentation.")

## 3. Load Vietnamese Appliance Test Fixtures

We load `data/fixtures/vietnamese_appliance_descriptions.csv` containing 18 curated appliance descriptions covering 6 major appliance categories.  
All records are explicitly tagged `TEST_FIXTURES_ONLY` with zero synthetic price labels.

In [ ]:
fixtures_path = Path("data/fixtures/vietnamese_appliance_descriptions.csv")
assert fixtures_path.exists(), f"Fixtures not found at {fixtures_path}"

df_fixtures = pd.read_csv(fixtures_path)
print(f"Total Test Fixtures: {len(df_fixtures)}")
print(f"Categories: {df_fixtures['category'].unique().tolist()}")
print(f"Brands: {df_fixtures['brand'].nunique()} unique brands")
print("\nCategory breakdown:")
print(df_fixtures['category'].value_counts())

# Sample preview
df_fixtures[['sample_id', 'category', 'brand', 'model_number', 'capacity', 'condition', 'warranty_status']].head(6)

## 4. PhoBERT Tokenization & Truncation Diagnostics

PhoBERT architecture details:
- Model checkpoint: `vinai/phobert-base`
- Max sequence length: 256 tokens (PhoBERT positional embedding limit: 258)
- Sentence separator: `</s>`
- Special tokens: `<s>` (BOS, id=0), `</s>` (EOS/SEP, id=2), `<pad>` (id=1), `<unk>` (id=3)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=False)
print(f"Tokenizer Vocab Size: {tokenizer.vocab_size:,}")
print(f"Model Max Length:     {tokenizer.model_max_length}")
print(f"BOS Token: '{tokenizer.bos_token}' (id={tokenizer.bos_token_id})")
print(f"EOS/SEP Token: '{tokenizer.eos_token}' (id={tokenizer.eos_token_id})")
print(f"PAD Token: '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")

# Check token lengths across all 18 fixtures
token_lengths = []
for idx, row in df_fixtures.iterrows():
    comb = processor.combine_title_description(row['title'], row['description'])
    tokens = tokenizer.encode(comb, add_special_tokens=True)
    token_lengths.append(len(tokens))

df_fixtures['token_count'] = token_lengths
print(f"\nMin Token Length:  {min(token_lengths)}")
print(f"Mean Token Length: {np.mean(token_lengths):.1f}")
print(f"Max Token Length:  {max(token_lengths)} (Limit is 256)")
print(f"Truncated Samples: {sum(l > 256 for l in token_lengths)} / {len(token_lengths)}")

## 5. PhoBERT Contextual Feature Extraction (768-d)

We instantiate `PhoBertFeatureExtractor` with:
- `model_name = 'vinai/phobert-base'`
- `pooling_strategy = 'mean'` (attention-mask weighted sum excluding padding tokens)
- `normalize_l2 = True` (unit-sphere projection for cosine distance consistency)
- `torch.inference_mode()` (frozen pretrained weights)

In [ ]:
extractor = PhoBertFeatureExtractor(
    model_name="vinai/phobert-base",
    device=device,
    pooling_strategy="mean",
    normalize_l2=True,
    max_length=256,
    segmenter="pyvi"
)

# Extract embeddings for all fixtures
results = extractor.extract_from_dataframe(df_fixtures, title_col="title", desc_col="description", batch_size=16)
embeddings = results["embeddings"]

print("=== EXTRACTION COMPLETE ===")
print(f"Embeddings Array Shape: {embeddings.shape}")
print(f"Embedding Data Type:    {embeddings.dtype}")
print(f"Number of Features:     {embeddings.shape[1]} (Expected: 768)")
print(f"Finite Values Check:    {np.isfinite(embeddings).all()} (No NaN / Inf)")

# Check L2 Norms
norms = np.linalg.norm(embeddings, axis=1)
print(f"L2 Norm Mean: {norms.mean():.4f}, Min: {norms.min():.4f}, Max: {norms.max():.4f}")

## 6. Semantic Similarity & Category Clustering Analysis

Because the embeddings are L2-normalized, the cosine similarity between item $i$ and item $j$ is simply the dot product:
$$\text{Sim}(i, j) = v_i \cdot v_j$$

Let us visualize the pairwise similarity matrix across the 18 fixture items to confirm that products in the same category cluster together naturally in PhoBERT's semantic space.

In [ ]:
# Compute pairwise cosine similarity matrix
similarity_matrix = np.dot(embeddings, embeddings.T)

# Build readable labels: Category - Brand Model
labels = [f"{row['category'][:4]} | {row['brand']} {row['model_number']}" for _, row in df_fixtures.iterrows()]

plt.figure(figsize=(12, 10))
sns.heatmap(
    similarity_matrix,
    xticklabels=labels,
    yticklabels=labels,
    cmap="viridis",
    vmin=0.4,
    vmax=1.0,
    annot=False
)
plt.title("Pairwise Semantic Cosine Similarity (vinai/phobert-base 768-d)", fontsize=14, fontweight="bold")
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

## 7. Verify Saved Artifacts & Reproducibility

We inspect the saved pipeline artifacts:
- `data/embeddings/phobert_demo_embeddings.npy`
- `data/embeddings/phobert_demo_mapping.csv`
- `data/embeddings/phobert_demo_metadata.json`

And verify deterministic reproducibility.

In [ ]:
saved_emb_path = Path("data/embeddings/phobert_demo_embeddings.npy")
saved_map_path = Path("data/embeddings/phobert_demo_mapping.csv")
saved_meta_path = Path("data/embeddings/phobert_demo_metadata.json")

assert saved_emb_path.exists(), f"Missing {saved_emb_path}"
assert saved_map_path.exists(), f"Missing {saved_map_path}"
assert saved_meta_path.exists(), f"Missing {saved_meta_path}"

loaded_embeddings = np.load(saved_emb_path)
df_mapping = pd.read_csv(saved_map_path)
with open(saved_meta_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("=== ARTIFACT VERIFICATION ===")
print(f"Loaded Embeddings Shape: {loaded_embeddings.shape}")
print(f"Loaded Mapping Rows:     {len(df_mapping)}")
print(f"Metadata Model:          {metadata['model_name']}")
print(f"Metadata Device:         {metadata['device']}")
print(f"Inference Duration:      {metadata['inference_duration_seconds']:.3f} s")
print(f"Throughput:              {metadata['throughput_samples_per_second']:.2f} samples/s")

# Check determinism
diff = np.abs(loaded_embeddings - embeddings).max()
print(f"Max absolute difference vs live run: {diff:.2e}")
assert diff < 1e-5, f"Numerical discrepancy detected: {diff}"
print("PASSED: Deterministic output verified across runs.")

## 8. Summary & Next Steps

### Pipeline Status: **PASS**
- Vietnamese text cleaning, normalization, and compound word segmentation: **Operational**
- Technical entity and model number preservation: **Verified**
- Pretrained PhoBERT feature extraction (768-d): **Verified**
- Attention-masked mean pooling excluding padding: **Verified**
- Truncation and metadata tracking: **Verified**

### Model Training Status: **BLOCKED**
- Multi-modal price regression model training is strictly BLOCKED pending authorized Chợ Tốt / SecondLife authentic Vietnamese listing dataset with real VND price labels.

### Downstream Multimodal Fusion Ready
- Text features: PhoBERT 768-d (`phobert_demo_embeddings.npy`)
- Image features: CLIP ViT-B/32 512-d (`clip_embeddings.npy` from TASK 04)
- Fusion Vector Dimension: $512 + 768 = 1280$ dimensions (+ structured tabular metadata)